# Notely AI Metric Analysis

This notebook moves from table exploration to product analytics reasoning.

The goal is to show how a data scientist would use the simulated Notely AI warehouse to answer PM-style questions before we connect an AI agent.

Key idea: the agent should eventually use trusted metric logic. This notebook is the human-readable version of that logic.

## 1. Setup

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

db_path = project_root / "data" / "notely_ai.sqlite"
conn = sqlite3.connect(db_path)

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn)

def read_sql_file(relative_path: str) -> str:
    return (project_root / relative_path).read_text()

print(db_path)
print("Database exists:", db_path.exists())

## 2. Product Context

Before analyzing metrics, inspect the known product calendar. In the future RAG version, these rows can become retrievable context for the agent.

In [ ]:
q("""
SELECT *
FROM product_calendar
ORDER BY calendar_date;
""")

## 3. Growth Overview

PM question: How has user acquisition trended over time?

In [ ]:
monthly_growth = q("""
SELECT
  substr(signup_date, 1, 7) AS signup_month,
  COUNT(*) AS new_users
FROM users
GROUP BY 1
ORDER BY 1;
""")
monthly_growth

In [ ]:
monthly_growth.plot(x="signup_month", y="new_users", kind="line", marker="o", figsize=(11, 4), title="Monthly New Users");

## 4. Activation Funnel

PM question: How many new users reach the first value moment?

Activation definition: within 7 days of signup, the user completes:

- `workspace_created`
- `recording_upload_completed`
- `ai_summary_generated`

In [ ]:
activation_funnel = q("""
WITH user_steps AS (
  SELECT
    u.user_id,
    MAX(CASE WHEN e.event_name = 'workspace_created' THEN 1 ELSE 0 END) AS has_workspace,
    MAX(CASE WHEN e.event_name = 'recording_upload_completed' THEN 1 ELSE 0 END) AS has_upload,
    MAX(CASE WHEN e.event_name = 'ai_summary_generated' THEN 1 ELSE 0 END) AS has_summary
  FROM users u
  LEFT JOIN events e
    ON u.user_id = e.user_id
   AND e.event_date BETWEEN u.signup_date AND date(u.signup_date, '+7 day')
  GROUP BY 1
)
SELECT 'signup_completed' AS funnel_step, COUNT(*) AS users, 1.000 AS step_rate FROM user_steps
UNION ALL
SELECT 'workspace_created', SUM(has_workspace), ROUND(1.0 * SUM(has_workspace) / COUNT(*), 3) FROM user_steps
UNION ALL
SELECT 'recording_upload_completed', SUM(CASE WHEN has_workspace = 1 AND has_upload = 1 THEN 1 ELSE 0 END), ROUND(1.0 * SUM(CASE WHEN has_workspace = 1 AND has_upload = 1 THEN 1 ELSE 0 END) / COUNT(*), 3) FROM user_steps
UNION ALL
SELECT 'ai_summary_generated', SUM(CASE WHEN has_workspace = 1 AND has_upload = 1 AND has_summary = 1 THEN 1 ELSE 0 END), ROUND(1.0 * SUM(CASE WHEN has_workspace = 1 AND has_upload = 1 AND has_summary = 1 THEN 1 ELSE 0 END) / COUNT(*), 3) FROM user_steps;
""")
activation_funnel

## 5. Storyline 1: iOS Upload Bug

PM question: Why did activation drop in mid-May?

Hypothesis from product context: an iOS upload retry bug started on 2026-05-10 and was fixed on 2026-05-17.

In [ ]:
weekly_activation = q("""
WITH signup_cohort AS (
  SELECT
    u.user_id,
    date(u.signup_date, 'weekday 1', '-7 days') AS signup_week,
    u.platform,
    MAX(CASE WHEN e.event_name = 'workspace_created' THEN 1 ELSE 0 END) AS has_workspace,
    MAX(CASE WHEN e.event_name = 'recording_upload_completed' THEN 1 ELSE 0 END) AS has_upload,
    MAX(CASE WHEN e.event_name = 'ai_summary_generated' THEN 1 ELSE 0 END) AS has_summary
  FROM users u
  LEFT JOIN events e
    ON u.user_id = e.user_id
   AND e.event_date BETWEEN u.signup_date AND date(u.signup_date, '+7 day')
  GROUP BY 1, 2, 3
)
SELECT
  signup_week,
  platform,
  COUNT(*) AS new_users,
  ROUND(AVG(CASE WHEN has_workspace = 1 AND has_upload = 1 AND has_summary = 1 THEN 1.0 ELSE 0.0 END), 3) AS activation_rate
FROM signup_cohort
WHERE signup_week BETWEEN '2026-04-20' AND '2026-06-01'
GROUP BY 1, 2
ORDER BY 1, 2;
""")
weekly_activation

In [ ]:
weekly_activation.pivot(index="signup_week", columns="platform", values="activation_rate").plot(figsize=(11, 4), marker="o", title="Activation Rate by Platform");

In [ ]:
q("""
SELECT
  event_date,
  platform,
  COUNT(*) AS upload_failures
FROM events
WHERE event_name = 'recording_upload_failed'
  AND event_date BETWEEN '2026-05-05' AND '2026-05-22'
GROUP BY 1, 2
ORDER BY 1, 2;
""")

Analysis takeaway: if activation drops mostly on iOS and upload failures spike during the incident window, the likely root cause is product reliability rather than demand quality.

## 6. Storyline 2: Paid Search Quality Decline

PM question: Why did signup volume increase but conversion decline after June 1?

Hypothesis: paid search spend increased, but the new traffic was lower intent.

In [ ]:
paid_search_quality = q("""
WITH user_outcomes AS (
  SELECT
    u.user_id,
    CASE WHEN u.signup_date >= '2026-06-01' THEN 'after_2026_06_01' ELSE 'before_2026_06_01' END AS period,
    u.acquisition_channel,
    MAX(CASE WHEN e.event_name = 'ai_summary_generated' THEN 1 ELSE 0 END) AS generated_summary,
    MAX(CASE WHEN e.event_name = 'subscription_started' THEN 1 ELSE 0 END) AS became_paid
  FROM users u
  LEFT JOIN events e ON u.user_id = e.user_id
  WHERE u.signup_date BETWEEN '2026-05-01' AND '2026-06-26'
  GROUP BY 1, 2, 3
)
SELECT
  period,
  acquisition_channel,
  COUNT(*) AS signups,
  ROUND(AVG(generated_summary), 3) AS summary_generation_rate,
  ROUND(AVG(became_paid), 3) AS paid_conversion_rate
FROM user_outcomes
GROUP BY 1, 2
ORDER BY 1, signups DESC;
""")
paid_search_quality

Analysis takeaway: separate volume from quality. Paid search can grow total signups while lowering overall conversion if it brings lower-intent users.

## 7. Storyline 3: Onboarding Flow Experiment

PM question: Did guided setup improve activation?

In [ ]:
q("""
WITH experiment_users AS (
  SELECT
    x.variant,
    u.user_id,
    MAX(CASE WHEN e.event_name = 'workspace_created' THEN 1 ELSE 0 END) AS has_workspace,
    MAX(CASE WHEN e.event_name = 'recording_upload_completed' THEN 1 ELSE 0 END) AS has_upload,
    MAX(CASE WHEN e.event_name = 'ai_summary_generated' THEN 1 ELSE 0 END) AS has_summary
  FROM experiments x
  JOIN users u ON x.user_id = u.user_id
  LEFT JOIN events e
    ON u.user_id = e.user_id
   AND e.event_date BETWEEN u.signup_date AND date(u.signup_date, '+7 day')
  WHERE x.experiment_name = 'onboarding_flow_v2'
  GROUP BY 1, 2
)
SELECT
  variant,
  COUNT(*) AS users,
  ROUND(AVG(has_workspace), 3) AS workspace_rate,
  ROUND(AVG(has_upload), 3) AS upload_rate,
  ROUND(AVG(CASE WHEN has_workspace = 1 AND has_upload = 1 AND has_summary = 1 THEN 1.0 ELSE 0.0 END), 3) AS activation_rate
FROM experiment_users
GROUP BY 1
ORDER BY 1;
""")

Analysis takeaway: the experiment should be read step-by-step, not only by the final activation metric. The lift may come from workspace creation, upload completion, or summary generation.

## 8. Storyline 4: Action Items and Retention

PM question: Are users who use action items more likely to retain?

In [ ]:
q("""
WITH base AS (
  SELECT
    u.user_id,
    u.signup_date,
    MAX(CASE WHEN e.event_name = 'action_items_generated' THEN 1 ELSE 0 END) AS used_action_items
  FROM users u
  LEFT JOIN events e ON u.user_id = e.user_id
  WHERE u.signup_date BETWEEN '2026-04-15' AND '2026-06-10'
  GROUP BY 1, 2
),
retention AS (
  SELECT
    b.user_id,
    b.used_action_items,
    CASE WHEN COUNT(e.event_id) > 0 THEN 1 ELSE 0 END AS retained_d7
  FROM base b
  LEFT JOIN events e
    ON b.user_id = e.user_id
   AND e.event_date BETWEEN date(b.signup_date, '+7 day') AND date(b.signup_date, '+13 day')
  GROUP BY 1, 2
)
SELECT
  used_action_items,
  COUNT(*) AS users,
  ROUND(AVG(retained_d7), 3) AS d7_retention
FROM retention
GROUP BY 1;
""")

Analysis takeaway: this is association, not causal proof. A strong retention difference is a good PM signal, but a real product decision may need experiment or causal analysis.

## 9. Storyline 5: Free Limit Change

PM question: Did reducing the Free limit improve monetization or hurt users?

In [ ]:
q("""
WITH cohorts AS (
  SELECT
    u.user_id,
    CASE WHEN u.signup_date >= '2026-05-25' THEN 'after_limit_change' ELSE 'before_limit_change' END AS period,
    MAX(CASE WHEN e.event_name = 'usage_limit_reached' THEN 1 ELSE 0 END) AS hit_usage_limit,
    MAX(CASE WHEN e.event_name = 'paywall_viewed' THEN 1 ELSE 0 END) AS saw_paywall,
    MAX(CASE WHEN e.event_name = 'subscription_started' THEN 1 ELSE 0 END) AS became_paid
  FROM users u
  LEFT JOIN events e ON u.user_id = e.user_id
  WHERE u.signup_date BETWEEN '2026-05-01' AND '2026-06-20'
  GROUP BY 1, 2
)
SELECT
  period,
  COUNT(*) AS users,
  ROUND(AVG(hit_usage_limit), 3) AS usage_limit_rate,
  ROUND(AVG(saw_paywall), 3) AS paywall_view_rate,
  ROUND(AVG(became_paid), 3) AS paid_conversion_rate
FROM cohorts
GROUP BY 1;
""")

Analysis takeaway: monetization changes should be evaluated with both conversion and retention/engagement. A paywall can increase paid conversion while harming Free user experience.

## 10. Weekly PM Report Prototype

The weekly report SQL is the bridge from manual analysis to the future Report Agent. The agent can later summarize this output into an executive-style report.

In [ ]:
weekly_report_sql = read_sql_file("sql/reports/weekly_pm_report.sql")
weekly_report = q(weekly_report_sql)
weekly_report

In [ ]:
weekly_report[["week_start", "new_users", "active_users", "activation_rate", "new_paid_users", "billing_persistence"]]

## 11. Where the Agent Comes In Later

At this point, we have a human-readable metric and analysis layer. Later, function calling can expose these as tools:

- `get_metric(metric_name, start_date, end_date, group_by)`
- `get_funnel(funnel_name, start_date, end_date, group_by)`
- `run_weekly_report(week_start)`
- `get_storyline_context(date_range, metric_name)`

The LLM should not invent metric definitions. It should use these trusted SQL patterns and then explain the results.

In [ ]:
conn.close()